# Лекция 7. Режимы и стратегии обучения — код

Код из лекции: обучение с учителем на 50 метках, автоэнкодеры (обычный, шумоподавляющий, разреженный), вариационный автоэнкодер и β-VAE, поиск аномалий, самообучение (предсказание поворота, SimCLR, маскированное моделирование), протоколы оценки представлений, частичное обучение (псевдометки, FixMatch, Mean Teacher), шумные и частичные метки, расписания curriculum и активное обучение.

Сквозной пример — цифры 8×8, у которых размечено только **50** картинок (по 5 каждой цифры), а остальные 1297 обучающих картинок считаются неразмеченными. В каждом разделе смотрим, сколько удаётся отыграть у полной разметки.

Ячейки выполняются **по порядку**: каждая следующая использует переменные из предыдущих. Проще всего — «Среда выполнения → Выполнить все». Видеокарта не нужна: весь ноутбук выполняется на процессоре примерно за полторы минуты, самые долгие ячейки — около 20 секунд. Время в выводе и последние знаки чисел у вас могут немного отличаться.

На своём компьютере понадобятся `pip install torch scikit-learn`.

## Глава 1. Обучение с учителем

### 1.3. Сколько стоит мало меток

Данные и инструменты. Картинки цифр делим на обучающие (75%) и тестовые (25%). Из обучающих размечаем только 5 картинок каждой цифры — это `X_lab`, `y_lab`; остальные — `X_unl`, их метки модели не показываем.

`train_supervised` — обычное обучение с учителем. Сравним модель на всех 1347 метках и только на 50: эти два числа — верхняя и нижняя границы для всей лекции.

In [ ]:
import copy
import time
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

torch.set_num_threads(4)                              # одинаковая скорость на разных машинах

# 1797 картинок 8×8, каждая вытянута в строку из 64 пикселей
X_all, y_all = load_digits(return_X_y=True)
X_all = torch.tensor(X_all, dtype=torch.float32) / 16   # яркость от 0 до 1
y_all = torch.tensor(y_all)
# 75% — обучающие, 25% — тест; доли цифр в обеих частях одинаковые (stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=0, stratify=y_all)

# размечено только 5 картинок каждой цифры; остальные обучающие — без меток
g = torch.Generator().manual_seed(0)
# для каждой цифры c: номера её картинок → случайная перестановка → первые 5
lab_idx = torch.cat([torch.nonzero(y_train == c).flatten()[torch.randperm(int((y_train == c).sum()), generator=g)[:5]]
                     for c in range(10)])
unl_mask = torch.ones(len(X_train), dtype=torch.bool)   # True — пример без метки
unl_mask[lab_idx] = False
X_lab, y_lab = X_train[lab_idx], y_train[lab_idx]     # 50 размеченных
X_unl = X_train[unl_mask]                             # метки y_train[unl_mask] модели не показываем

def make_mlp(d_in=64, width=128, d_out=10):
    """Сеть из двух скрытых слоёв: 64 пикселя → 128 → 128 → d_out выходов (логитов)."""
    return nn.Sequential(nn.Linear(d_in, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU(), nn.Linear(width, d_out))

def accuracy(model, X=X_test, y=y_test):
    """Доля верных ответов: класс с наибольшим логитом сравниваем с меткой."""
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

def train_supervised(X, y, epochs=300, lr=1e-3, wd=1e-4, augment=None, seed=0):
    """Обычное обучение с учителем: кросс-энтропия, Adam, батчи по 64."""
    torch.manual_seed(seed)                           # одинаковый старт для честного сравнения
    model = make_mlp()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    for epoch in range(epochs):
        order = torch.randperm(len(X))                # новый порядок примеров в каждой эпохе
        for i in range(0, len(X), 64):
            b = order[i:i + 64]                       # номера примеров очередного батча
            xb = augment(X[b]) if augment else X[b]   # аугментация — заново на каждом шаге
            loss = F.cross_entropy(model(xb), y[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model

print("обучающих:", len(X_train), "| размеченных:", len(X_lab), "| без меток:", len(X_unl), "| тест:", len(X_test))
full = train_supervised(X_train, y_train, epochs=60)  # верхняя граница: все метки
few = train_supervised(X_lab, y_lab)                  # нижняя граница: только 50 меток
print(f"все {len(X_train)} меток: точность {accuracy(full):.3f}")
print(f"только 50 меток:   точность {accuracy(few):.3f}")

**Что получилось.** Все метки дают 0.971, 50 меток — 0.847. Любой метод, использующий неразмеченные картинки, должен превзойти 0.847.

## Глава 2. Обучение без учителя: автоэнкодеры

### 2.2. Автоэнкодер

**Автоэнкодер** сжимает картинку из 64 пикселей в **код** из 8 чисел и восстанавливает её обратно; обучается без меток. Качество кода проверяем **линейным пробником** — логистической регрессией на 50 размеченных примерах.

In [ ]:
def probe(features):
    """Линейный пробник: логистическая регрессия на 50 размеченных примерах, точность на тесте."""
    with torch.no_grad():                             # кодировщик заморожен: градиенты не нужны
        f_lab, f_test = features(X_lab).numpy(), features(X_test).numpy()
    clf = LogisticRegression(max_iter=2000).fit(f_lab, y_lab.numpy())
    return clf.score(f_test, y_test.numpy())          # score — точность на тесте

class AutoEncoder(nn.Module):
    def __init__(self, d_latent=8):
        super().__init__()
        # кодировщик: 64 пикселя → 128 → код из d_latent чисел
        self.encoder = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, d_latent))
        # декодер: код → 128 → 64 пикселя; сигмоида держит яркость в [0, 1]
        self.decoder = nn.Sequential(nn.Linear(d_latent, 128), nn.ReLU(), nn.Linear(128, 64), nn.Sigmoid())

    def forward(self, x):
        return self.decoder(self.encoder(x))          # реконструкция x̂ = D(E(x))

def train_ae(model, loss_fn, epochs=200, corrupt=None, seed=0):
    """Обучение без меток: на вход — X_train (возможно, испорченный), цель — исходная картинка."""
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(epochs):
        order = torch.randperm(len(X_train))
        for i in range(0, len(X_train), 64):
            x = X_train[order[i:i + 64]]              # метки не используются вовсе
            x_in = corrupt(x) if corrupt else x       # вход: испорченный (раздел 2.3) или чистый
            loss = loss_fn(model, x_in, x)            # сравниваем всегда с чистым x
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model

# потеря реконструкции: средний квадрат разницы между реконструкцией и чистой картинкой
mse_loss = lambda model, x_in, x: F.mse_loss(model(x_in), x)
torch.manual_seed(0)
ae = train_ae(AutoEncoder(), mse_loss)
with torch.no_grad():
    print(f"ошибка реконструкции (MSE) на тесте: {F.mse_loss(ae(X_test), X_test):.4f}")
print(f"линейный пробник на пикселях (64 числа): {probe(lambda x: x):.3f}")   # признаки — сами пиксели
print(f"линейный пробник на кодах (8 чисел):     {probe(ae.encoder):.3f}")    # признаки — коды кодировщика

**Что получилось.** Реконструкция хорошая, но пробник на кодах (0.796) **хуже**, чем на пикселях (0.847): автоэнкодер хранит всё нужное для восстановления картинки — толщину и наклон штрихов, — а не только то, что отличает цифры.

Четыре потери реконструкции на четырёх пикселях, где последний восстановлен с грубой ошибкой 0.8.

In [ ]:
x = torch.tensor([0.2, 0.5, 0.9, 0.1])                # «исходные» четыре пикселя
x_hat = torch.tensor([0.3, 0.4, 0.8, 0.9])            # последняя координата — грубая ошибка
# ошибки: 0.1, −0.1, −0.1 и 0.8; все потери — средние по четырём пикселям
for name, value in [("MSE", F.mse_loss(x_hat, x)), ("MAE", F.l1_loss(x_hat, x)),
                    ("Huber (δ = 0.1)", F.huber_loss(x_hat, x, delta=0.1)),
                    ("BCE", F.binary_cross_entropy(x_hat, x))]:
    print(f"{name:16s} {value:.4f}")
# доля MSE, которую дал один выброс: его квадрат ошибки / сумма квадратов всех ошибок
print("вклад выброса в MSE:", round(((x_hat - x)[3] ** 2 / ((x_hat - x) ** 2).sum()).item(), 2))

**Что получилось.** Один выброс дал 96% всей MSE. Важнее градиент: у MSE выброс тянет модель в 8 раз сильнее маленькой ошибки, у MAE и потери Хьюбера с $\delta = 0.1$ — так же, как маленькая ошибка.

### 2.3. Шумоподавляющий автоэнкодер

**Шумоподавляющий автоэнкодер** получает испорченную картинку (шум и 30% обнулённых пикселей), а восстанавливать должен чистую. `F.mse_loss(a, b)` — средний квадрат разности по всем пикселям: сравниваем с чистым тестом испорченные картинки и результаты обоих автоэнкодеров.

In [ ]:
def corrupt(x, p=0.3, sigma=0.2):
    """Гауссов шум и обнуление доли p пикселей; результат обрезаем в [0, 1]."""
    keep = (torch.rand_like(x) > p).float()           # 1 — пиксель остаётся, 0 — обнуляется (~30%)
    return ((x + sigma * torch.randn_like(x)) * keep).clamp(0, 1)

dae = train_ae(AutoEncoder(), mse_loss, corrupt=corrupt)   # вход испорчен, цель — чистая картинка
torch.manual_seed(1)
X_noisy = corrupt(X_test)                             # испорченный тест — одинаковый для обеих моделей
with torch.no_grad():
    print(f"испорченный тест против чистого:     MSE {F.mse_loss(X_noisy, X_test):.4f}")
    print(f"обычный AE восстанавливает:          MSE {F.mse_loss(ae(X_noisy), X_test):.4f}")
    print(f"шумоподавляющий AE восстанавливает:  MSE {F.mse_loss(dae(X_noisy), X_test):.4f}")
print(f"линейный пробник на кодах DAE: {probe(dae.encoder):.3f}")   # коды считаем по чистым картинкам

* Корень из MSE — средняя ошибка в единицах яркости: испорченные картинки ≈ 0.29, обычный автоэнкодер ≈ 0.24, шумоподавляющий ≈ 0.17.
* Пробник на кодах шумоподавляющего автоэнкодера — **0.876**, выше пикселей: шум разрушает детали, и кодировщик учится сохранять форму цифры.

### 2.4. Разреженный автоэнкодер

**Разреженный автоэнкодер**: код из 128 чисел (вдвое больше входа) и L1-штраф, который заставляет большинство чисел кода быть нулями.

In [ ]:
def sparse_loss(lam):
    """Реконструкция плюс L1-штраф на код с весом lam: среднее |z| по всем координатам."""
    def loss(model, x_in, x):
        z = model.encoder(x_in)
        return F.mse_loss(model.decoder(z), x) + lam * z.abs().mean()
    return loss

class WideAE(AutoEncoder):
    """Переполненный автоэнкодер: код из 128 чисел — вдвое длиннее входа."""
    def __init__(self):
        super().__init__(d_latent=128)
        self.encoder.append(nn.ReLU())                # коды неотрицательные: 0 — «нейрон молчит»

for lam in [0.0, 0.1]:                                # без штрафа и со штрафом
    sae = train_ae(WideAE(), sparse_loss(lam))
    with torch.no_grad():
        z = sae.encoder(X_test)                       # коды тестовых картинок: 450 × 128
        rec = F.mse_loss(sae.decoder(z), X_test)
    active = (z > 0.01).float().mean().item()         # доля заметно ненулевых координат кода
    print(f"λ = {lam}: MSE {rec:.4f} | активных нейронов кода {active:.0%} | пробник {probe(sae.encoder):.3f}")

**Что получилось.** Со штрафом включается лишь 14% нейронов кода: сеть оставляет включённым нейрон, только если он окупает штраф. Пробник почти не изменился — разреженность здесь помогает интерпретируемости, а не точности.

### 2.5. Вариационный автоэнкодер

**Вариационный автоэнкодер** кодирует картинку распределением: кодировщик выдаёт среднее $\mu$ и $\log \sigma^2$, код выбирается **репараметризацией** $z = \mu + \sigma \cdot \varepsilon$. Потеря — минус ELBO: реконструкция плюс KL. В конце сгенерируем цифру из случайного кода и напечатаем её крупными «пикселями».

In [ ]:
class VAE(nn.Module):
    def __init__(self, d_latent=8):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(64, 128), nn.ReLU())                # общая часть кодировщика
        self.mu, self.logvar = nn.Linear(128, d_latent), nn.Linear(128, d_latent)   # две головы: μ и log σ²
        self.decoder = nn.Sequential(nn.Linear(d_latent, 128), nn.ReLU(), nn.Linear(128, 64), nn.Sigmoid())

    def forward(self, x):
        h = self.body(x)
        mu, logvar = self.mu(h), self.logvar(h)       # параметры распределения q(z|x)
        eps = torch.randn_like(mu)                    # шум не зависит от параметров
        z = mu + torch.exp(0.5 * logvar) * eps        # репараметризация: z = μ + σ·ε, σ = exp(½·log σ²)
        return self.decoder(z), mu, logvar

def vae_loss(beta=1.0):
    """Минус ELBO: реконструкция плюс β·KL, в среднем на одну картинку."""
    def loss(model, x_in, x):
        x_hat, mu, logvar = model(x_in)
        rec = F.binary_cross_entropy(x_hat, x, reduction="sum") / len(x)       # −log p(x|z) на картинку
        kl = 0.5 * (mu ** 2 + logvar.exp() - 1 - logvar).sum() / len(x)        # KL(q(z|x) || N(0, I))
        return rec + beta * kl
    return loss

def vae_report(model):
    """Реконструкция и KL на тесте, число «живых» размерностей и пробник на μ."""
    with torch.no_grad():
        h = model.body(X_test)
        mu, logvar = model.mu(h), model.logvar(h)
        # реконструкцию считаем по среднему μ, без случайного шума
        rec = F.binary_cross_entropy(model.decoder(mu), X_test, reduction="sum") / len(X_test)
        kl_dim = (0.5 * (mu ** 2 + logvar.exp() - 1 - logvar)).mean(dim=0)   # KL по каждой размерности
    feats = lambda x: model.mu(model.body(x))         # признаки для пробника — среднее μ
    # размерность «живая», если её KL заметно больше нуля: она несёт информацию о входе
    return f"реконструкция {rec:5.1f} | KL {kl_dim.sum():5.1f} | живых размерностей {int((kl_dim > 0.1).sum())} из 8 | пробник {probe(feats):.3f}"

vae = train_ae(VAE(), vae_loss(1.0))
print("VAE, β = 1:", vae_report(vae))

def show(img):
    """Печатает картинку 8×8 крупными «пикселями»: чем ярче пиксель, тем плотнее символ."""
    chars = " ░▒▓█"                                   # 5 уровней яркости: от пустого до сплошного
    for row in img.view(8, 8):
        line = "".join(chars[min(int(v * 5), 4)] * 4 for v in row)   # пиксель — 4 символа в ширину
        print(line)
        print(line)                                   # и 2 строки в высоту: так он выглядит квадратным

torch.manual_seed(3)
with torch.no_grad():
    sample = vae.decoder(torch.randn(1, 8))           # новая картинка из случайного z ~ N(0, I)
print("\nкартинка, сгенерированная из случайного z:")
show(sample[0])
with torch.no_grad():
    p = F.softmax(full(sample), dim=1)[0]             # что видит в ней классификатор из раздела 1.3
print(f"классификатор, обученный на всех метках: цифра {p.argmax().item()} с вероятностью {p.max():.2f}")

* Живых размерностей 5 из 8: остальные совпали с $\mathcal{N}(0, I)$ — модели хватило пяти.
* Из **случайного** кода получилась правдоподобная цифра — классификатор узнаёт двойку. Она размыта: декодер VAE «усредняет» похожие начертания.

### 2.6. β-VAE и проблемы обучения

**β-VAE**: вес KL $\beta = 0.1$ и $\beta = 4$.

In [ ]:
for beta in [0.1, 4.0]:                               # слабая и сильная регуляризация кодов
    print(f"β = {beta:3}:", vae_report(train_ae(VAE(), vae_loss(beta))))

* При $\beta = 0.1$ все размерности живые, реконструкция лучше.
* При $\beta = 4$ KL ноль и живых размерностей нет — **коллапс апостериорного распределения**: код перестал зависеть от входа, пробник упал до 0.36.

### 2.7. Применения автоэнкодеров

**Поиск аномалий**: автоэнкодер обучаем только на цифрах 0–8, а девятки считаем аномалиями. Оценка аномальности — ошибка реконструкции.

In [ ]:
from sklearn.metrics import roc_auc_score

normal = y_train != 9                                 # обучаем только на цифрах 0–8
X_norm_train = X_train[normal]
torch.manual_seed(0)
ae_norm = AutoEncoder()
opt = torch.optim.Adam(ae_norm.parameters(), lr=1e-3)
for epoch in range(200):                              # тот же цикл, что в train_ae, но на «нормальных» данных
    order = torch.randperm(len(X_norm_train))
    for i in range(0, len(X_norm_train), 64):
        x = X_norm_train[order[i:i + 64]]
        loss = F.mse_loss(ae_norm(x), x)
        opt.zero_grad()
        loss.backward()
        opt.step()

def score(x):
    """Оценка аномальности: ошибка реконструкции каждого примера."""
    with torch.no_grad():
        return ((ae_norm(x) - x) ** 2).mean(dim=1)    # средний квадрат ошибки по 64 пикселям

s_test = score(X_test)
threshold = score(X_norm_train).quantile(0.95)        # 95% нормальных примеров ниже порога
is_nine = (y_test == 9)                               # «аномалии» в тесте — девятки
# ROC-AUC не зависит от порога: насколько оценки девяток выше оценок остальных цифр
print(f"ROC-AUC «девятка или нет»: {roc_auc_score(is_nine.numpy(), s_test.numpy()):.3f}")
print(f"выше порога: девяток {(s_test[is_nine] > threshold).float().mean():.0%}, "
      f"остальных цифр {(s_test[~is_nine] > threshold).float().mean():.0%}")
torch.manual_seed(0)
noise_imgs = torch.rand(100, 64)                      # совсем не цифры: случайный шум
print(f"выше порога: картинок-шума {(score(noise_imgs) > threshold).float().mean():.0%}")

**Что получилось.** ROC-AUC 0.92: девятки получают ошибку выше других цифр. Шум — весь выше порога. Но выше порога и 17% обычных тестовых цифр, хотя на обучающих — 5%: порог нужно выбирать на отдельных нормальных данных, которых модель не видела.

## Глава 3. Самообучение

### 3.2. Предтекстовые задачи

**Предтекстовая задача** самообучения: угадать, на сколько повёрнута картинка (0°, 90°, 180° или 270°). Затем — пробник на признаках этой сети.

In [ ]:
def rotate(x, k):
    """Поворот картинок 8×8 на k·90°."""
    return torch.rot90(x.view(-1, 8, 8), k, dims=(1, 2)).reshape(-1, 64)

torch.manual_seed(0)
rot_net = make_mlp(d_out=4)                           # 4 выхода: поворот на 0°, 90°, 180° или 270°
opt = torch.optim.Adam(rot_net.parameters(), lr=1e-3)
for epoch in range(60):
    order = torch.randperm(len(X_train))
    for i in range(0, len(X_train), 64):
        x = X_train[order[i:i + 64]]
        k = torch.randint(4, (len(x),))               # метка предтекстовой задачи — номер поворота
        # поворачиваем группами с одинаковым k и в том же порядке собираем метки
        x_rot = torch.cat([rotate(x[k == r], r) for r in range(4)])
        k_sorted = torch.cat([k[k == r] for r in range(4)])
        loss = F.cross_entropy(rot_net(x_rot), k_sorted)
        opt.zero_grad()
        loss.backward()
        opt.step()

with torch.no_grad():
    # каждая тестовая картинка во всех четырёх поворотах
    k = torch.arange(4).repeat_interleave(len(X_test))
    x_rot = torch.cat([rotate(X_test, r) for r in range(4)])
    print(f"угадывает поворот на тесте: {(rot_net(x_rot).argmax(1) == k).float().mean():.3f}")
# rot_net[:4] — сеть без последнего слоя: признаки второго скрытого слоя (128 чисел)
print(f"пробник на скрытом слое сети поворотов: {probe(rot_net[:4]):.3f}")

**Что получилось.** Поворот сеть угадывает в 95% случаев, но пробник — лишь 0.547: для цифр 8×8 задача слишком лёгкая и не требует узнавать цифру.

### 3.3. Контрастивное обучение

**SimCLR**: две случайные аугментации одной картинки должны получить близкие представления, а версии разных картинок — далёкие. Потеря — InfoNCE для пар (NT-Xent). Здесь же определены `weak_aug` и `strong_aug` — они нужны и дальше. Обучение — около 20 секунд.

In [ ]:
def shift(x, max_shift=1):
    """Случайный сдвиг каждой картинки на −1..1 пиксель по каждой оси (края заполняются нулями)."""
    img = F.pad(x.view(-1, 1, 8, 8), (max_shift,) * 4)    # рамка из нулей: картинки 10×10
    dx, dy = torch.randint(0, 2 * max_shift + 1, (2, len(x)))   # смещение окна 8×8 для каждой картинки
    return torch.stack([img[i, 0, dy[i]:dy[i] + 8, dx[i]:dx[i] + 8] for i in range(len(x))]).reshape(-1, 64)

def weak_aug(x):
    """Слабая аугментация — только сдвиг."""
    return shift(x)

def strong_aug(x):
    """Сдвиг, шум, случайная яркость и выключение 20% пикселей."""
    x = shift(x) * (0.6 + 0.8 * torch.rand(len(x), 1))    # яркость всей картинки ×0.6…1.4
    x = x + 0.1 * torch.randn_like(x)                     # гауссов шум
    return (x * (torch.rand_like(x) > 0.2)).clamp(0, 1)   # выключаем ~20% пикселей

class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        # кодировщик: его признаки (128 чисел) потом проверяем пробником
        self.encoder = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU())
        # проектор: на его выходе (32 числа) считают потерю, после обучения его выбрасывают
        self.projector = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 32))

    def forward(self, x):
        return F.normalize(self.projector(self.encoder(x)), dim=1)   # длина 1: скалярное произведение = косинус

def nt_xent(z1, z2, tau=0.5):
    """InfoNCE для пар: позитив для z1[i] — z2[i], негативы — остальные 2N − 2 вида батча."""
    z = torch.cat([z1, z2])                           # 2N векторов: сначала первые версии, потом вторые
    n = len(z1)
    sim = (z @ z.T / tau).fill_diagonal_(-float("inf"))   # косинусы всех пар / τ; сам с собой — исключаем
    # для строки i < N позитив стоит в столбце i + N, для строки i ≥ N — в столбце i − N
    target = torch.cat([torch.arange(n, 2 * n), torch.arange(n)])
    return F.cross_entropy(sim, target)               # «угадай позитив среди 2N − 1 кандидатов»

t = time.perf_counter()
torch.manual_seed(0)
simclr = SimCLR()
opt = torch.optim.Adam(simclr.parameters(), lr=1e-3)
for epoch in range(1000):
    order = torch.randperm(len(X_train))
    for i in range(0, len(X_train), 256):             # батч 256: больше негативов
        x = X_train[order[i:i + 256]]
        # две независимые сильные аугментации одних и тех же картинок
        loss = nt_xent(simclr(strong_aug(x)), simclr(strong_aug(x)))
        opt.zero_grad()
        loss.backward()
        opt.step()
print(f"обучение SimCLR: {time.perf_counter() - t:.0f} с, последняя потеря {loss.item():.2f}")
print(f"пробник на кодировщике SimCLR: {probe(simclr.encoder):.3f}")

**Что получилось.** Пробник на кодировщике SimCLR — **0.887**: лучше пикселей (0.847) и автоэнкодера (0.796).

### 3.5. Маскированное моделирование

**Маскированное моделирование**: прячем половину пикселей и учим сеть их восстанавливать.

In [ ]:
class MaskedAE(nn.Module):
    def __init__(self):
        super().__init__()
        # вход — 128 чисел: 64 пикселя (спрятанные обнулены) и 64 флага «пиксель виден»
        self.encoder = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU())
        self.decoder = nn.Linear(128, 64)             # предсказывает все 64 пикселя

    def features(self, x):
        return self.encoder(torch.cat([x, torch.ones_like(x)], dim=1))   # без маски: видны все пиксели

torch.manual_seed(0)
mae = MaskedAE()
opt = torch.optim.Adam(mae.parameters(), lr=1e-3)
for epoch in range(100):
    order = torch.randperm(len(X_train))
    for i in range(0, len(X_train), 64):
        x = X_train[order[i:i + 64]]
        visible = (torch.rand_like(x) > 0.5).float()  # прячем половину пикселей
        # на вход — картинка с обнулёнными спрятанными пикселями и сама маска
        pred = torch.sigmoid(mae.decoder(mae.encoder(torch.cat([x * visible, visible], dim=1))))
        hidden = 1 - visible
        loss = ((pred - x) ** 2 * hidden).sum() / hidden.sum()           # ошибка только на спрятанных
        opt.zero_grad()
        loss.backward()
        opt.step()
print(f"ошибка на спрятанных пикселях: {loss.item():.4f}")
print(f"пробник на кодировщике маскированной модели: {probe(mae.features):.3f}")

**Что получилось.** Пробник — 0.873, почти как у SimCLR: чтобы угадать спрятанную половину цифры, нужно понять, что это за цифра.

### 3.6. Как оценивать самообучение

Три протокола оценки представлений: линейный пробник, 1-NN (ближайший сосед) и дообучение всей сети на 50 примерах.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

def knn(features, k=1):
    """k ближайших соседей среди 50 размеченных; векторы нормированы — сравниваем направления."""
    with torch.no_grad():
        f_lab, f_test = F.normalize(features(X_lab), dim=1), F.normalize(features(X_test), dim=1)
    return KNeighborsClassifier(k).fit(f_lab.numpy(), y_lab.numpy()).score(f_test.numpy(), y_test.numpy())

def finetune(encoder, epochs=100, lr=1e-3, seed=0):
    """Дообучение: копия кодировщика + линейная голова, учится вся сеть на 50 примерах."""
    torch.manual_seed(seed)
    model = nn.Sequential(copy.deepcopy(encoder), nn.Linear(128, 10))   # копия — чтобы не испортить оригинал
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        loss = F.cross_entropy(model(weak_aug(X_lab)), y_lab)   # все 50 примеров — один батч
        opt.zero_grad()
        loss.backward()
        opt.step()
    return accuracy(model)

print(f"{'признаки':22s} пробник  1-NN    дообучение")
print(f"{'пиксели':22s} {probe(lambda x: x):.3f}   {knn(lambda x: x):.3f}   —")
print(f"{'автоэнкодер':22s} {probe(ae.encoder):.3f}   {knn(ae.encoder):.3f}   —")
torch.manual_seed(0)
# случайный кодировщик — та же сеть без обучения: нижняя граница
for name, enc in [("случайный кодировщик", SimCLR().encoder), ("SimCLR", simclr.encoder)]:
    print(f"{name:22s} {probe(enc):.3f}   {knn(enc):.3f}   {finetune(enc):.3f}")

* Случайный кодировщик хуже пикселей — выигрыш SimCLR дало именно обучение.
* У SimCLR лучшие пробник и 1-NN.
* Дообучение SimCLR на 50 примерах хуже его пробника: градиенты случайной головы портят хорошие признаки.

## Глава 4. Частичное обучение

### 4.2. Псевдометки и минимизация энтропии

**Частичное обучение**. Сначала честный базовый уровень — 50 меток со слабыми аугментациями, затем один раунд **псевдометок**: уверенные предсказания на неразмеченных картинках становятся метками.

In [ ]:
base = train_supervised(X_lab, y_lab, augment=weak_aug)   # 50 меток + сдвиги
print(f"50 меток + слабые аугментации: {accuracy(base):.3f}")

with torch.no_grad():
    prob = F.softmax(base(X_unl), dim=1)              # вероятности классов для неразмеченных
conf, pseudo = prob.max(dim=1)                        # уверенность и предсказанный класс
keep = conf > 0.95                                    # берём только уверенные
y_unl_hidden = y_train[unl_mask]                      # спрятанные метки — только для проверки
print(f"уверенных: {keep.float().mean():.0%}, из них верно: {(pseudo[keep] == y_unl_hidden[keep]).float().mean():.0%}")
# заново обучаем на 50 настоящих метках и отобранных псевдометках
pl = train_supervised(torch.cat([X_lab, X_unl[keep]]), torch.cat([y_lab, pseudo[keep]]), epochs=150, augment=weak_aug)
print(f"после одного раунда псевдометок: {accuracy(pl):.3f}")

**Что получилось.** Псевдометки почти безошибочны (99% верных), но точность не выросла: уверенно размечаются лёгкие примеры, из которых модель нового не узнаёт.

### 4.4. FixMatch и MixMatch

**FixMatch**: псевдометку ставит слабо искажённая версия неразмеченного примера (если модель уверена), а учится модель на сильно искажённой. Печатаем долю уверенных примеров и точность по ходу обучения.

In [ ]:
def fixmatch(steps=3000, tau=0.95, lam=1.0, mu=7, seed=0):
    """Слабая аугментация даёт псевдометку, сильная — пример, на котором её учат."""
    torch.manual_seed(seed)
    model = make_mlp()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    for step in range(steps):
        x_l = X_lab                                   # все 50 размеченных в каждом шаге
        x_u = X_unl[torch.randint(len(X_unl), (mu * len(x_l),))]   # 7 · 50 = 350 случайных неразмеченных
        loss_s = F.cross_entropy(model(weak_aug(x_l)), y_lab)      # обычная потеря на размеченных
        with torch.no_grad():                         # псевдометка — константа, градиент через неё не идёт
            p = F.softmax(model(weak_aug(x_u)), dim=1)
        conf, y_hat = p.max(dim=1)                    # уверенность и псевдометка по слабой версии
        mask = (conf >= tau).float()                  # учим только на уверенных
        # кросс-энтропия на СИЛЬНОЙ версии с псевдометкой; неуверенные обнуляются маской
        loss_u = (F.cross_entropy(model(strong_aug(x_u)), y_hat, reduction="none") * mask).mean()
        loss = loss_s + lam * loss_u
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step + 1 in (300, 1000, 3000):             # печатаем ход обучения
            print(f"шаг {step + 1:4d}: уверенных {mask.mean():.0%} | точность {accuracy(model):.3f}")
    return model

t = time.perf_counter()
fm = fixmatch()
print(f"FixMatch: {time.perf_counter() - t:.0f} с")

**Что получилось.** Доля уверенных растёт с 27% до 88%, точность — до **0.938**: без единой новой метки отыграно три четверти разрыва до полной разметки.

**Mean Teacher**: учитель — скользящее среднее весов ученика; потеря согласованности требует, чтобы ученик на сильно искажённой версии отвечал так же, как учитель на слабо искажённой.

In [ ]:
def mean_teacher(steps=3000, alpha=0.99, lam=10.0, seed=0):
    """Ученик учится градиентом, учитель — скользящее среднее его весов и цель для согласованности."""
    torch.manual_seed(seed)
    student = make_mlp()
    teacher = copy.deepcopy(student).requires_grad_(False)   # учителя градиентом не обучают
    opt = torch.optim.Adam(student.parameters(), lr=1e-3, weight_decay=1e-4)
    for step in range(steps):
        x_u = X_unl[torch.randint(len(X_unl), (350,))]
        loss_s = F.cross_entropy(student(weak_aug(X_lab)), y_lab)
        p_student = F.softmax(student(strong_aug(x_u)), dim=1)   # ученик видит сильно искажённую версию
        with torch.no_grad():
            p_teacher = F.softmax(teacher(weak_aug(x_u)), dim=1) # учитель — слабо искажённую
        ramp = min(1.0, step / 1000)                  # вес согласованности растёт постепенно
        # согласованность: MSE между векторами вероятностей ученика и учителя
        loss = loss_s + lam * ramp * F.mse_loss(p_student, p_teacher)
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():                         # учитель — скользящее среднее весов ученика
            for pt, ps in zip(teacher.parameters(), student.parameters()):
                pt.mul_(alpha).add_((1 - alpha) * ps)   # θ_teacher ← α·θ_teacher + (1 − α)·θ_student
    return student, teacher

student, teacher = mean_teacher()
print(f"Mean Teacher: ученик {accuracy(student):.3f} | учитель {accuracy(teacher):.3f}")

**Что получилось.** Mean Teacher — 0.940, на уровне FixMatch.

## Глава 5. Слабая разметка

### 5.3. Отбор надёжных примеров

**Шумные метки**: 40% меток всех обучающих картинок заменены случайным другим классом. Сравним кросс-энтропию, label smoothing, GCE и отбор примеров с малой потерей (small-loss). Тест — с правильными метками.

In [ ]:
def add_noise(y, rate, seed=0):
    """Симметричный шум: доля rate меток заменяется случайным другим классом."""
    g = torch.Generator().manual_seed(seed)
    flip = torch.rand(len(y), generator=g) < rate     # какие метки портим
    other = (y + torch.randint(1, 10, (len(y),), generator=g)) % 10   # сдвиг на 1..9 — всегда другой класс
    return torch.where(flip, other, y), flip

y_noisy, flipped = add_noise(y_train, 0.4)
print(f"неверных меток: {flipped.float().mean():.0%}")

def gce(logits, y, q=0.7):
    """Generalized Cross Entropy: (1 − p_y^q) / q."""
    p_y = F.softmax(logits, dim=1).gather(1, y[:, None]).squeeze(1)   # вероятность класса из метки
    return ((1 - p_y ** q) / q).mean()

def train_noisy(loss_fn, epochs=150, keep=None, seed=0):
    """Обучение на шумных метках; keep — доля примеров батча для small-loss (None — все)."""
    torch.manual_seed(seed)
    model = make_mlp()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = {}
    for epoch in range(epochs):
        order = torch.randperm(len(X_train))
        for i in range(0, len(X_train), 64):
            b = order[i:i + 64]
            losses = loss_fn(model(X_train[b]), y_noisy[b])          # потери по каждому примеру
            if keep and epoch >= 10:                                 # small-loss: учимся на самых «лёгких»
                losses = losses.sort().values[:int(keep * len(b))]   # доля keep с наименьшей потерей
            loss = losses.mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        if epoch + 1 in (20, 150):
            history[epoch + 1] = accuracy(model)      # точность на тесте с правильными метками
    with torch.no_grad():
        # какую долю испорченных примеров модель относит к неверному классу из метки — «выучила шум»
        fit_noise = (model(X_train[flipped]).argmax(1) == y_noisy[flipped]).float().mean().item()
    return history, fit_noise

# три потери, возвращающие значение для каждого примера (reduction="none")
ce = lambda logits, y: F.cross_entropy(logits, y, reduction="none")
smooth = lambda logits, y: F.cross_entropy(logits, y, reduction="none", label_smoothing=0.1)
gce_each = lambda logits, y: (1 - F.softmax(logits, dim=1).gather(1, y[:, None]).squeeze(1) ** 0.7) / 0.7
for name, fn, keep in [("кросс-энтропия", ce, None), ("label smoothing 0.1", smooth, None),
                       ("GCE, q = 0.7", gce_each, None), ("small-loss, 60%", ce, 0.6)]:
    h, fit = train_noisy(fn, keep=keep)
    print(f"{name:20s} тест: эпоха 20 — {h[20]:.3f}, эпоха 150 — {h[150]:.3f} | выучено неверных меток {fit:.0%}")

* Кросс-энтропия на 20-й эпохе — 0.956, но к 150-й падает до 0.778: модель запомнила больше половины неверных меток.
* Label smoothing не помогает.
* GCE и small-loss держат 0.93 и выучили лишь 8% неверных меток.

### 5.4. Частичные и групповые метки

**Частичные метки**: для каждой картинки известен лишь набор возможных классов. Сравним мягкую цель «поровну на все классы набора» и потерю «хоть один из набора» $-\log \sum_{c \in S} p_c$.

In [ ]:
g = torch.Generator().manual_seed(0)
candidates = torch.rand(len(y_train), 10, generator=g) < 0.3   # каждый ложный класс попадает в набор с вероятностью 0.3
candidates[torch.arange(len(y_train)), y_train] = True         # истинный класс всегда в наборе
print(f"в среднем классов в наборе: {candidates.float().sum(1).mean():.1f}")

def train_partial(mode, epochs=100, seed=0):
    """Обучение по наборам кандидатов: «равномерно» или «сумма по набору»."""
    torch.manual_seed(seed)
    model = make_mlp()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    S = candidates.float()                            # 1 — класс в наборе, 0 — нет
    for epoch in range(epochs):
        order = torch.randperm(len(X_train))
        for i in range(0, len(X_train), 64):
            b = order[i:i + 64]
            logp = F.log_softmax(model(X_train[b]), dim=1)   # логарифмы вероятностей классов
            if mode == "равномерно":                  # мягкая цель: поровну на все классы набора
                loss = -(S[b] / S[b].sum(1, keepdim=True) * logp).sum(1).mean()
            else:                                     # «хоть один из набора»: −log Σ p_c по набору
                # классы вне набора — −∞, logsumexp складывает вероятности только кандидатов
                loss = -torch.logsumexp(logp.masked_fill(~candidates[b], -float("inf")), dim=1).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
    return accuracy(model)

for mode in ["равномерно", "сумма по набору"]:
    print(f"{mode:16s} точность {train_partial(mode):.3f}")

**Что получилось.** «Хоть один из набора» — 0.967, почти как с точными метками; равномерная цель — 0.858: она учит модель заведомо неправильному ответу.

## Глава 6. Curriculum и активное обучение

### 6.1. Curriculum: от простого к сложному

Три функции темпа curriculum — доля доступных данных $c(t)$ по ходу обучения.

In [ ]:
T = 10                                                # длительность обучения в условных шагах
# доля доступных данных c(t) для трёх функций темпа; печатаем t = 0, 2, 4, …, 10
for name, c in [("линейное, c₀ = 0.2", lambda t: min(0.2 + 0.08 * t, 1.0)),
                ("экспоненциальное", lambda t: 1 - (1 - 0.2) * torch.exp(torch.tensor(-0.3 * t)).item()),
                ("корень, r = 2", lambda t: max((t / T) ** 0.5, 0.2))]:
    print(f"{name:20s}", " ".join(f"{c(t):.2f}" for t in range(0, T + 1, 2)))

**Что получилось.** К середине обучения экспоненциальное расписание открывает 76–87% данных, линейное — 52–68%.

### 6.2. Активное обучение

**Активное обучение**: начинаем с 20 случайных размеченных примеров и за 8 раундов добавляем по 10, выбирая их случайно или по неуверенности модели (least confidence, margin, энтропия). Каждую стратегию запускаем с трёх стартовых наборов.

In [ ]:
# каждая стратегия по вероятностям p (примеры × классы) возвращает приоритет: больше — раньше размечать
def least_confidence(p):
    return 1 - p.max(dim=1).values                    # чем ниже вероятность лучшего класса, тем выше приоритет

def margin(p):
    top2 = p.topk(2, dim=1).values                    # две наибольшие вероятности
    return -(top2[:, 0] - top2[:, 1])                 # чем меньше разрыв, тем выше приоритет

def entropy(p):
    return -(p * p.clamp_min(1e-12).log()).sum(dim=1)   # clamp — чтобы не брать log 0

def active_learning(strategy, rounds=8, per_round=10, seed=0):
    """Цикл активного обучения; возвращает точность на тесте после каждого раунда."""
    g = torch.Generator().manual_seed(seed)
    labeled = torch.randperm(len(X_train), generator=g)[:20].tolist()   # старт: 20 случайных примеров
    curve = []
    for r in range(rounds + 1):
        # 1) обучаем модель на текущих размеченных и записываем точку кривой обучения
        clf = LogisticRegression(max_iter=2000).fit(X_train[labeled].numpy(), y_train[labeled].numpy())
        curve.append(clf.score(X_test.numpy(), y_test.numpy()))
        if r == rounds:
            break
        # 2) пул — все ещё не размеченные обучающие примеры
        pool = torch.tensor([i for i in range(len(X_train)) if i not in set(labeled)])
        if strategy is None:                          # случайный выбор — базовая линия
            chosen = pool[torch.randperm(len(pool), generator=g)[:per_round]]
        else:
            p = torch.tensor(clf.predict_proba(X_train[pool].numpy()))
            chosen = pool[strategy(p).topk(per_round).indices]   # 10 примеров с наибольшим приоритетом
        labeled += chosen.tolist()                    # «оракул» размечает выбранные примеры
    return curve

print("размечено:        ", "  ".join(f"{20 + 10 * r:5d}" for r in range(9)))
for name, s in [("случайно", None), ("least confidence", least_confidence), ("margin", margin), ("энтропия", entropy)]:
    curves = [active_learning(s, seed=k) for k in range(3)]   # три разных стартовых набора
    mean = [sum(c[r] for c in curves) / 3 for r in range(9)]  # средняя кривая
    print(f"{name:18s}", "  ".join(f"{a:.3f}" for a in mean), f"| площадь {sum(mean) / 9:.3f}")

* **Margin** — лучшая стратегия: точность 0.89 уже на 60 метках вместо 100 у случайного выбора.
* Least confidence и энтропия по площади под кривой хуже случайного выбора: на 10 классах они выбирают нетипичные, плохо написанные цифры.
* Стратегию активного обучения всегда сравнивают со случайным выбором.